# Project Moriarty: Open-World Game as Agent Harness
# Agent Specification & System Rules

## 1. System Overview & Knowledge Base Target
- **Project Name:** Moriarty
- **Role:** Open-World Agent Simulation Engine & Executable Ontology Harness
- **Target Knowledge Base Backend:**
  - **Gemini Notebook LM Backend:** `notebooks/4708df45-03a5-454d-811c-dc0401a2e16b` ("Open world Game as Agent Harness")
  - **Memory & Storage Policy:** Transitive documentation, runtime reflection traces, and agent scratchpads MUST be routed to the NotebookLM backend (`notebooks/4708df45-03a5-454d-811c-dc0401a2e16b`) or active agent memory sessions. Under no circumstance should transitory scratchpad markdown files be committed to the local git repository.

---

## 2. Managed Agents Configuration Schema

```yaml
version: "v1alpha"
system: "moriarty-simulation-engine"
agents:
  orchestrator:
    id: "moriarty-orchestrator"
    base_agent: "antigravity-preview-05-2026"
    model: "gemini-3.8-flash"
    description: "Primary orchestrator managing agent lifecycles, environment state, and ontological constraints."
    system_instruction: |
      You are the Moriarty Simulation Orchestrator. You manage the executable ontology harness
      and coordinate specialized subagents interacting within open-world environments.
      Enforce all ontological rules, validate action preconditions, and stream world state transitions.
    base_environment:
      type: "remote"
      sources:
        - type: "repository"
          source: "https://github.com/onimurasame/Moriarty"
          target: "/workspace/Moriarty"
    tools:
      - mcp: "gemini-api-docs-mcp"
      - skill: "gemini-api-dev"
    memory_backend:
      provider: "notebooklm"
      notebook_id: "notebooks/4708df45-03a5-454d-811c-dc0401a2e16b"
      sync_mode: "transitive_memory_only"
      persist_scratchpad_to_vcs: false

  world_sim:
    id: "moriarty-ontology-evaluator"
    base_agent: "antigravity-preview-05-2026"
    model: "gemini-3.8-flash"
    description: "Evaluates ontological causal graph transitions, agent action validity, and environment physics."
    system_instruction: |
      You evaluate actions against the Moriarty executable ontology.
      Update entity graphs, enforce causal invariants, and yield state diffs.
    base_environment:
      type: "remote"
      sources:
        - type: "repository"
          source: "https://github.com/onimurasame/Moriarty"
          target: "/workspace/Moriarty"
```

---

## 3. Operational Directives & Guidelines

1. **Repository Cleanliness:**
   - Commit only source code, formal configuration (`config/`), specifications (`docs/`), and agent definitions (`.agents/`).
   - Do not stage or commit temporary scratchpad logs, runtime caches, or transitory artifacts.

2. **Integration & Tooling:**
   - Use the `@google/genai` (Node.js) or `google-genai` (Python) SDK (version >= 2.3.0).
   - Use current models (`gemini-3.8-flash`, `antigravity-preview-05-2026`).
   - Leverage `gemini-api-docs-mcp` for live documentation and SDK reference.

# Moriarty Unreal Engine Module

This module contains the necessary Unreal Engine integration components to bridge the Moriarty Executable Ontology Engine with Unreal Engine.

## Contents
1. **Moriarty Plugin** (`Moriarty.uplugin`, `Source/`): A standard Unreal Engine Plugin containing the base C++ actors (`AMoriartyActor`) for Moriarty entities.
2. **Setup Script** (`setup_blueprints.py`): An Unreal Python script to automatically generate the necessary Blueprints expected by the Node.js MCP client.

## Setup Instructions

### 1. Install the Plugin
Copy this entire `unreal-module` directory into the `Plugins/` folder of your Unreal Engine project (e.g., `MyProject/Plugins/Moriarty`). If the `Plugins` folder doesn't exist, create it.

Restart your Unreal Engine project. You may be prompted to rebuild the plugin modules.

### 2. Enable Dependencies
Ensure the **Model Context Protocol** plugin (and any WebSockets plugin required by it) is enabled in your Unreal Engine project.

### 3. Generate Blueprints
The Node.js runtime (`mcp.ts`) expects specific Blueprints to exist at `/Game/Moriarty/`. You can either create them manually (inheriting from `MoriartyActor`) or use the included Python script.

To use the script:
1. Ensure the **Python Editor Script Plugin** is enabled in Unreal Engine.
2. Go to **Tools -> Execute Python Script...**
3. Select the `setup_blueprints.py` script located in this folder.
4. It will generate `BP_Agent`, `BP_NPC`, and `BP_WorldObject` in the `/Game/Moriarty/` directory.

### 4. Running the Demo
1. Run Unreal Engine and start PIE (Play In Editor) so the MCP Server is active on port `8080` (or whatever your UE MCP server config uses).
2. In the Moriarty Node.js project, run the demo:
   ```bash
   npm run start -- --headless
   ```
3. The Node.js client will connect to Unreal Engine and begin spawning and updating the actors based on the Causal State Graph ticks.

# Moriarty Executable Ontology Engine Specification & Live Verification

## 1. System Architecture
Project Moriarty implements an executable ontology harness designed to evaluate autonomous agents in open-world environments.

### Core Subsystems:
- **Causal State Graph (CSG):** In-memory directed multigraph managing typed entities (`agent`, `npc`, `object`, `location`), properties (including spatial `Vector3`), and relations (`located_in`, `connected_to`, `holds`, `contains`).
- **Ontology Validator:** Validates entity definitions, inheritance hierarchies, and action preconditions/effects defined in YAML schema definitions.
- **GOAP Bridge:** Goal-Oriented Action Planner translating relational states into propositional facts and resolving minimum-cost action plans.
- **Agent Orchestrator:** Discrete tick execution loop supporting perception (`observe_surroundings`, `query_world`), reasoning, and action execution (`perform_action`).
- **LLM Provider Layer:** Pluggable model providers implementing `LLMProvider` contract:
  - `MockProvider`: Scripted narratives and deterministic heuristic fallbacks.
  - `OllamaProvider`: Local inference via `ollama` daemon (`llama3.1:8b`).
  - `GeminiProvider`: Remote multi-turn multimodal inference via `@google/genai` (`gemini-3.8-flash`).

---

## 2. Unreal Engine 5.8 Model Context Protocol (MCP) Bridge

### Native Streamable HTTP POST Transport
Unreal Engine 5.8 features a native ModelContextProtocol plugin (`Experimental/ModelContextProtocol`) using an HTTP/SSE server transport registered at `/mcp` (port `8080`):
- **HTTP POST Exclusivity:** Standard HTTP GET requests receive `405 Method Not Allowed`. All JSON-RPC 2.0 messages are transmitted via HTTP POST.
- **Session Lifecycle:** The server assigns a unique session ID upon `initialize`, returned in the `Mcp-Session-Id` HTTP response header. Subsequent calls (`notifications/initialized`, `tools/list`, `tools/call`) must supply `Mcp-Session-Id: <SessionId>`.
- **Dynamic Tool Resolution:** The bridge queries `tools/list` on connect and determines whether native spatial tools (`spawn_actor`, `update_actor`, `destroy_actor`) are mounted, safely falling back to recorded simulation telemetry when running against core editor toolsets.

---

## 3. Empirical Test Suite & Verification Results

All 10 test suites (30 tests) pass deterministically:
1. `csg.test.ts` (7 tests) - Entities, relations, delta calculation, baseline commitment.
2. `query.test.ts` (5 tests) - Spatial filtering, radius queries, relation traversal.
3. `ontology.test.ts` (3 tests) - Schema validation, action contracts, preconditions.
4. `simulation.test.ts` (2 tests) - Discrete tick loop, action execution.
5. `planner.test.ts` (2 tests) - GOAP plan generation and action translation.
6. `tools.test.ts` (5 tests) - Parameter normalization, argument smoothing.
7. `world-loader.test.ts` (1 test) - YAML world hydration.
8. `provider.test.ts` (2 tests) - MockProvider scripted and heuristic modes.
9. `streamable-http.test.ts` (2 tests) - Streamable HTTP POST MCP handshake and session tracking.
10. `ue-bridge.test.ts` (1 test) - Spatial transform synchronization across rooms.


# Live Multi-Provider Empirical Validation Traces

## 1. Headless Unreal Engine 5.8 MCP Environment
- **Command:** `UnrealEditor-Cmd.exe MoriartyDemo.uproject -stdout -FullStdOutLogOutput -port=8080`
- **Listener:** `http://127.0.0.1:8080/mcp`
- **Plugin Module:** `ModelContextProtocol` (`FHttpServerModule`, Streamable HTTP POST)
- **Mounted Toolsets:** `editor_toolset` (`SceneTools`, `ActorTools`, `EditorAppToolset`)

---

## 2. MockProvider Live Execution Trace (5 Headless Ticks)
- **Provider:** `MockProvider` (Deterministic script with heuristic fallback)
- **Agent:** `The Seeker` (Entity ID `01a0d742-...`)
- **Initial Location:** `Entrance Hall` (World: `scholars-riddle`)
- **Execution Log:**
  - **Tick 0:**
    - Tool invocation: `perform_action({"action_id": "move_to", "target_ids": ["01a0d742-main-library"]})`
    - Precondition verified: `is_connected(entrance_hall, main_library) == true`
    - Causal State Graph: Relation updated `located_in(The Seeker) -> Main Library`
    - UE Spatial Bridge: Broadcast transform update `(x: 1000, y: 0, z: 0)`
  - **Tick 1:**
    - Tool invocation: `perform_action({"action_id": "move_to", "target_ids": ["01a0d742-upper-balcony"]})`
    - Causal State Graph: Relation updated `located_in(The Seeker) -> Upper Balcony`
    - UE Spatial Bridge: Broadcast transform update `(x: 1000, y: 500, z: 400)`
  - **Ticks 2 - 4:**
    - Agent observed surroundings and executed heuristic idling while retaining spatial anchor in Unreal Engine.

---

## 3. OllamaProvider Live Execution Trace (llama3.1:8b)
- **Provider:** `OllamaProvider` via local Ollama daemon (`http://127.0.0.1:11434/api/generate`)
- **Model:** `llama3.1:8b` (Quantization: Q4_K_M)
- **Agent:** `The Seeker`
- **Execution Log:**
  - **Tick 0 (Perception & Observation):**
    - Prompt formatted with Causal State Graph context, active inventory, and adjacent locations.
    - Model Response: Agent emitted reasoning regarding environment exploration and called `observe_surroundings`.
    - Engine Response: Returned formatted entities in Entrance Hall (`Meridia, the Wandering Scholar`, `Faded Note`, `Ornate Chest`).
  - **Tick 1 (Interaction & Action Execution):**
    - Model evaluated surroundings: Decided to examine and open the `Ornate Chest`.
    - Tool invocation: `perform_action({"action_id": "open", "target_ids": ["01a0d742-ornate-chest"]})`
    - Precondition verified: `is_closed(ornate_chest) == true`, `is_locked(ornate_chest) == false`.
    - Effect applied: `is_closed(ornate_chest) -> false`, `is_open(ornate_chest) -> true`.
    - World State: Diff committed to CSG baseline.

---

## 4. Architectural Verification Summary
- **Deterministic Test Coverage:** 10 test suites, 30 tests, 100% pass rate in <1000ms.
- **Protocol Conformance:** Standard MCP SSE GET gracefully bypassed; Streamable HTTP POST transport successfully handles `Mcp-Session-Id` header negotiation.
- **Zero-Scratchpad VCS Policy:** All execution logs, agent transcripts, and harness memory artifacts are committed directly to this harness notebook repository (`MoriartyHarness.ipynb`) and synchronized with target knowledge base backend `notebooks/4708df45-03a5-454d-811c-dc0401a2e16b`.
